# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rahmanislamzada/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# Retrieve HF_TOKEN securely from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    print("HF_TOKEN successfully retrieved.")
except Exception:
    hf_token = ""
    print("HF_TOKEN not found in Secrets, running with fallback sample data.")

con = duckdb.connect(':memory:')

# Generate robust sample data for Week 3 Data Contract verification
np.random.seed(42)
n = 1000
df_fact = pd.DataFrame({
    'client_hash_id': [f"client_{i%5}" for i in range(n)],
    'content_hash_id': [f"page_{i%100}" for i in range(n)],
    'snapshot_date': pd.date_range(start='2026-03-01', periods=n, freq='h'),
    'gsc_clicks': np.random.poisson(lam=12, size=n),
    'gsc_impressions': np.random.poisson(lam=250, size=n),
    'gsc_sum_position': np.random.uniform(300, 2500, size=n),
    'is_available': np.random.choice([True, False], size=n, p=[0.85, 0.15])
})

con.register('fact_daily', df_fact)

print("=== QUERY 1: Grain Verification (1 Row = 1 Page per Client) ===")
q1 = con.execute("""
    SELECT client_hash_id, content_hash_id, COUNT(*) as record_count
    FROM fact_daily
    GROUP BY client_hash_id, content_hash_id
    LIMIT 3
""").df()
display(q1)

print("\n=== QUERY 2: Slice Row Count & Date Span (Mid-Panel Month: 2026-03) ===")
q2 = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        MIN(snapshot_date) as start_date,
        MAX(snapshot_date) as end_date
    FROM fact_daily
    WHERE snapshot_date >= '2026-03-01' AND snapshot_date < '2026-04-01'
""").df()
display(q2)

print("\n=== QUERY 3: Availability Check (Filtering IS TRUE) ===")
q3 = con.execute("""
    SELECT
        COUNT(*) as total_before_filter,
        COUNT(CASE WHEN is_available IS TRUE THEN 1 END) as surviving_rows,
        ROUND(COUNT(CASE WHEN is_available IS TRUE THEN 1 END)::FLOAT / COUNT(*) * 100, 2) as survival_rate_pct
    FROM fact_daily
""").df()
display(q3)

print("\n=== 5 FEATURE FRAME & LEAKAGE EXPERIMENT ===")
df_features = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        -- Feature 1: Historical Clicks
        SUM(gsc_clicks) AS feat_hist_clicks,
        -- Feature 2: Historical Impressions
        SUM(gsc_impressions) AS feat_hist_impressions,
        -- Feature 3: Calculated CTR
        AVG(gsc_clicks::FLOAT / NULLIF(gsc_impressions, 0)) AS feat_avg_ctr,
        -- Feature 4: Average Search Position
        AVG(gsc_sum_position::FLOAT / NULLIF(gsc_impressions, 0)) AS feat_avg_position,
        -- Feature 5: Traffic Volatility (StdDev of Clicks)
        COALESCE(STDDEV_SAMP(gsc_clicks), 0) AS feat_click_volatility,

        -- TRAP: Label-derived Column (Target Leakage Column)
        SUM(gsc_clicks) * 1.05 AS LEAKED_future_clicks_target
    FROM fact_daily
    WHERE is_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print("Feature Frame created with 5 clean features + 1 Leaked Column.")
display(df_features.head(3))

# Removing the leaked column to keep the honest baseline
df_clean_features = df_features.drop(columns=['LEAKED_future_clicks_target'])
print("\n[LEAKAGE REMOVED] Clean Feature Frame ready for honest baseline modeling:")
display(df_clean_features.head(3))

HF_TOKEN successfully retrieved.
=== QUERY 1: Grain Verification (1 Row = 1 Page per Client) ===


,client_hash_id,content_hash_id,record_count
0,client_2,page_2,10
1,client_3,page_3,10
2,client_4,page_4,10



=== QUERY 2: Slice Row Count & Date Span (Mid-Panel Month: 2026-03) ===


,total_rows,start_date,end_date
0,744,2026-03-01,2026-03-31 23:00:00



=== QUERY 3: Availability Check (Filtering IS TRUE) ===


,total_before_filter,surviving_rows,survival_rate_pct
0,1000,844,84.400002



=== 5 FEATURE FRAME & LEAKAGE EXPERIMENT ===
Feature Frame created with 5 clean features + 1 Leaked Column.


,client_hash_id,content_hash_id,feat_hist_clicks,feat_hist_impressions,feat_avg_ctr,feat_avg_position,feat_click_volatility,LEAKED_future_clicks_target
0,client_0,page_0,86.0,1745.0,0.049800,6.127679,2.811541,90.3
1,client_1,page_1,122.0,2222.0,0.055020,6.359355,2.403701,128.1
2,client_1,page_6,104.0,2248.0,0.046211,6.829607,2.877113,109.2



[LEAKAGE REMOVED] Clean Feature Frame ready for honest baseline modeling:


,client_hash_id,content_hash_id,feat_hist_clicks,feat_hist_impressions,feat_avg_ctr,feat_avg_position,feat_click_volatility
0,client_0,page_0,86.0,1745.0,0.049800,6.127679,2.811541
1,client_1,page_1,122.0,2222.0,0.055020,6.359355,2.403701
2,client_1,page_6,104.0,2248.0,0.046211,6.829607,2.877113


## 2. Fields: feature / label / context / excluded

Features: feat_hist_clicks, feat_hist_impressions, feat_avg_ctr, feat_avg_position, feat_click_volatility

Label: target_total_clicks (Next month total GSC search clicks)

Context: client_hash_id, content_hash_id, snapshot_date

Excluded: Pages with < 50 total impressions in observation window (Reason: excludes extreme low-traffic noise/outliers).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

Data Limitation: The dataset assumes stable search engine indexing algorithms; sudden Google core updates can shift page ranking positions without prior historical pattern warnings.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.